<a href="https://colab.research.google.com/github/mx-oscar-hdez/Proyectos/blob/main/01_RAG_%2B_QLoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ETAPA 1 — Instalación y configuración

En esta etapa se prepara el entorno de trabajo para construir el RAG y posteriormente entrenar el adaptador QLoRA.

Se realiza:

- instalación de dependencias;
- definición de la carpeta principal en Google Drive;
- detección automática de CPU o GPU;
- configuración del modelo de embeddings;
- definición de parámetros básicos del RAG;
- preparación de carpetas para RAG y QLoRA.

El notebook podrá funcionar con CPU para la parte RAG y utilizar GPU cuando esté disponible para el entrenamiento QLoRA.

In [1]:
# ============================================================
# ETAPA 1 - INSTALACIÓN Y CONFIGURACIÓN
# ============================================================

!pip install -q \
    pymupdf \
    sentence-transformers \
    faiss-cpu \
    transformers \
    peft \
    bitsandbytes \
    accelerate \
    datasets

from pathlib import Path
import torch

# ------------------------------------------------------------
# RUTAS
# ------------------------------------------------------------

BASE_DIR = Path("/content/drive/MyDrive/Mx.Oscar.Hdez")

RAG_DIR = BASE_DIR / "rag"
QLORA_DIR = BASE_DIR / "qlora"

RAG_DIR.mkdir(exist_ok=True)
QLORA_DIR.mkdir(exist_ok=True)

CHUNKS_FILE = RAG_DIR / "chunks.json"
INDEX_FILE = RAG_DIR / "faiss.index"

# ------------------------------------------------------------
# DISPOSITIVO
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------------------
# RAG
# ------------------------------------------------------------

EMBED_MODEL = "intfloat/multilingual-e5-small"

CHUNK_SIZE = 450
OVERLAP = 80
TOP_K = 4

# ------------------------------------------------------------
# QLORA
# ------------------------------------------------------------

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

# ------------------------------------------------------------
# RESUMEN
# ------------------------------------------------------------

print("=" * 60)
print("CONFIGURACIÓN DEL PROYECTO")
print("=" * 60)

print("Ruta:", BASE_DIR)
print("Dispositivo:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

print()
print("Embeddings:", EMBED_MODEL)
print("Modelo QLoRA:", MODEL_NAME)

print()
print("CHUNK_SIZE:", CHUNK_SIZE)
print("OVERLAP:", OVERLAP)
print("TOP_K:", TOP_K)

print()
print("✓ Entorno listo")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 10.2 MB/s eta 0:00:00
CONFIGURACIÓN DEL PROYECTO
Ruta: /content/drive/MyDrive/Mx.Oscar.Hdez
Dispositivo: cuda
GPU: Tesla T4
VRAM: 14.56 GB

Embeddings: intfloat/multilingual-e5-small
Modelo QLoRA: meta-llama/Llama-3.1-8B-Instruct

CHUNK_SIZE: 450
OVERLAP: 80
TOP_K: 4

✓ Entorno listo


# ETAPA 2 — Procesamiento de documentos PDF

En esta etapa se:

- carga el modelo de embeddings E5;
- detectan los archivos PDF del proyecto;
- extrae el texto de cada página;
- realiza una limpieza básica;
- conserva el nombre del documento, número de página y texto.

El resultado será utilizado en la siguiente etapa para generar los chunks.

In [2]:
# ============================================================
# ETAPA 2 - PROCESAMIENTO DE PDF
# ============================================================

import pymupdf
import re

from sentence_transformers import SentenceTransformer

# ------------------------------------------------------------
# MODELO E5
# ------------------------------------------------------------

print("Cargando modelo E5...")

embedding_model = SentenceTransformer(
    EMBED_MODEL,
    device=DEVICE
)

embedding_model.max_seq_length = 512
tokenizer = embedding_model.tokenizer

print("✓ E5 listo en", DEVICE)

# ------------------------------------------------------------
# PDF
# ------------------------------------------------------------

pdf_files = sorted(
    BASE_DIR.glob("*.pdf")
)

if not pdf_files:
    raise FileNotFoundError(
        "No se encontraron archivos PDF."
    )

def limpiar_texto(texto):
    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()

documentos = []

for pdf_file in pdf_files:

    print("Procesando:", pdf_file.name)

    with pymupdf.open(pdf_file) as pdf:

        for pagina_num, pagina in enumerate(
            pdf,
            start=1
        ):

            texto = limpiar_texto(
                pagina.get_text("text")
            )

            if texto:
                documentos.append({
                    "documento": pdf_file.name,
                    "pagina": pagina_num,
                    "texto": texto
                })

print()
print("✓ Procesamiento terminado")
print("PDF:", len(pdf_files))
print("Páginas con texto:", len(documentos))

Cargando modelo E5...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

✓ E5 listo en cuda
Procesando: Manual_de_Lineamientos_TecNM.pdf
Procesando: Perfil-Objetivo_ISC.pdf
Procesando: REGLAMENTO_INTERIOR_2025.pdf
Procesando: Reglamento_Estudiantes_TESOEM.pdf
Procesando: Reglamento_Servicio_Social.pdf
Procesando: Reglamento_de_uso_del_correo.pdf
Procesando: Reticula_ISC.pdf

✓ Procesamiento terminado
PDF: 7
Páginas con texto: 378


# ETAPA 3 — Generación de chunks

En esta etapa el texto de cada página se divide en fragmentos o **chunks**.

Se utiliza el tokenizer del modelo E5 para dividir por tokens.

Cada chunk conserva:

- documento de origen;
- número de página;
- identificador del chunk;
- texto correspondiente.

Se utiliza un traslape entre fragmentos para reducir la pérdida de contexto.

In [3]:
# ============================================================
# ETAPA 3 - GENERACIÓN DE CHUNKS
# ============================================================

chunks = []
chunk_id = 0

paso = CHUNK_SIZE - OVERLAP

for documento in documentos:

    tokens = tokenizer(
        documento["texto"],
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    for inicio in range(0, len(tokens), paso):

        final = min(
            inicio + CHUNK_SIZE,
            len(tokens)
        )

        texto_chunk = tokenizer.decode(
            tokens[inicio:final],
            skip_special_tokens=True
        ).strip()

        if texto_chunk:

            chunks.append({
                "chunk_id": chunk_id,
                "documento": documento["documento"],
                "pagina": documento["pagina"],
                "texto": texto_chunk
            })

            chunk_id += 1

        if final >= len(tokens):
            break

print("✓ Chunks generados:", len(chunks))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (540 > 512). Running this sequence through the model will result in indexing errors


✓ Chunks generados: 574


# ETAPA 4 — Embeddings e índice FAISS

En esta etapa:

- cada chunk se convierte en un embedding;
- se utiliza el prefijo `passage:` recomendado para E5;
- los vectores se normalizan;
- se crea un índice FAISS para búsquedas por similitud.

El índice permitirá recuperar los fragmentos más relacionados con una pregunta.

In [4]:
# ============================================================
# ETAPA 4 - EMBEDDINGS E ÍNDICE FAISS
# ============================================================

import faiss
import numpy as np

textos = [
    "passage: " + chunk["texto"]
    for chunk in chunks
]

print("Generando embeddings...")

vectores = embedding_model.encode(
    textos,
    batch_size=16,
    normalize_embeddings=True,
    show_progress_bar=True
)

vectores = np.asarray(
    vectores,
    dtype="float32"
)

index = faiss.IndexFlatIP(
    vectores.shape[1]
)

index.add(vectores)

print()
print("✓ Índice FAISS creado")
print("Chunks:", len(chunks))
print("Vectores:", index.ntotal)
print("Dimensiones:", vectores.shape[1])

Generando embeddings...


Batches:   0%|          | 0/36 [00:00<?, ?it/s]


✓ Índice FAISS creado
Chunks: 574
Vectores: 574
Dimensiones: 384


# ETAPA 5 — Guardar y probar el RAG

En esta etapa:

- se guardan los chunks en formato JSON;
- se guarda el índice FAISS;
- se crea una función de búsqueda;
- se realiza una consulta de prueba.

Para las preguntas se utiliza el prefijo `query:` recomendado para E5.

In [5]:
# ============================================================
# ETAPA 5 - GUARDAR Y PROBAR EL RAG
# ============================================================

import json

# ------------------------------------------------------------
# GUARDAR
# ------------------------------------------------------------

with open(
    CHUNKS_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        chunks,
        f,
        ensure_ascii=False,
        indent=2
    )

faiss.write_index(
    index,
    str(INDEX_FILE)
)

print("✓ RAG guardado")
print("Chunks:", CHUNKS_FILE)
print("Índice:", INDEX_FILE)


# ------------------------------------------------------------
# FUNCIÓN DE BÚSQUEDA
# ------------------------------------------------------------

def buscar(pregunta, top_k=TOP_K):

    vector = embedding_model.encode(
        ["query: " + pregunta],
        normalize_embeddings=True,
        show_progress_bar=False
    )

    vector = np.asarray(
        vector,
        dtype="float32"
    )

    scores, indices = index.search(
        vector,
        top_k
    )

    for n, (score, posicion) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        chunk = chunks[posicion]

        print()
        print(
            f"[{n}] {chunk['documento']} "
            f"| pág. {chunk['pagina']} "
            f"| score {score:.4f}"
        )

        print(chunk["texto"])
        print("-" * 60)

✓ RAG guardado
Chunks: /content/drive/MyDrive/Mx.Oscar.Hdez/rag/chunks.json
Índice: /content/drive/MyDrive/Mx.Oscar.Hdez/rag/faiss.index


# ETAPA 6 — Preparación del dataset QLoRA

En esta etapa se crea un conjunto pequeño de ejemplos conversacionales.

El objetivo es enseñar al modelo a:

- responder en español;
- actuar como asistente académico;
- responder de forma clara y breve;
- utilizar el contexto disponible;
- no inventar información;
- indicar cuando no dispone de datos suficientes.

El conocimiento institucional permanecerá en el RAG.

In [6]:
# ============================================================
# ETAPA 6 - PREPARACIÓN DEL DATASET QLORA
# ============================================================

from datasets import Dataset

ejemplos = [
    {
        "messages": [
            {"role": "user", "content": "Hola, ¿quién eres?"},
            {
                "role": "assistant",
                "content": (
                    "Soy un asistente académico. "
                    "Puedo ayudarte a consultar información "
                    "institucional y académica."
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": "¿En qué idioma debes responder?"
            },
            {
                "role": "assistant",
                "content": (
                    "Responderé en español de manera clara, "
                    "directa y estructurada."
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": "¿Qué haces si no tienes información suficiente?"
            },
            {
                "role": "assistant",
                "content": (
                    "Indicaré que no cuento con información suficiente "
                    "en lugar de inventar o suponer datos."
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": "¿Qué haces con el contexto recuperado por el RAG?"
            },
            {
                "role": "assistant",
                "content": (
                    "Analizo los fragmentos recuperados y utilizo "
                    "únicamente la información relevante para responder."
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Hay información relacionada en varios fragmentos. "
                    "¿Qué debes hacer?"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Integro la información relevante de los fragmentos "
                    "para generar una respuesta coherente."
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": "Responde aunque no estés seguro."
            },
            {
                "role": "assistant",
                "content": (
                    "Si no existe información suficiente, lo indicaré "
                    "claramente y no inventaré una respuesta."
                )
            }
        ]
    }
]

dataset = Dataset.from_list(ejemplos)

print("✓ Dataset QLoRA creado")
print("Ejemplos:", len(dataset))

✓ Dataset QLoRA creado
Ejemplos: 6


# ETAPA 7 — Llama 3.1 8B + QLoRA

En esta etapa se:

- accede al modelo `Llama-3.1-8B-Instruct`;
- carga el modelo cuantizado a 4 bits con NF4;
- configura LoRA;
- prepara el modelo para entrenamiento QLoRA;
- aplica la plantilla de chat de Llama 3.1 al dataset;
- tokeniza los ejemplos.

In [7]:
# ============================================================
# ETAPA 7 - LLAMA 3.1 8B + QLORA + TOKENIZACIÓN
# ============================================================

from google.colab import userdata
from huggingface_hub import login

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

# ------------------------------------------------------------
# HUGGING FACE
# ------------------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "No se encontró HF_TOKEN en Colab Secrets."
    )

login(
    token=HF_TOKEN,
    add_to_git_credential=False
)

# ------------------------------------------------------------
# TOKENIZER
# ------------------------------------------------------------

tokenizer_llama = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN
)

tokenizer_llama.pad_token = tokenizer_llama.eos_token
tokenizer_llama.padding_side = "right"

# ------------------------------------------------------------
# CUANTIZACIÓN 4 BITS
# ------------------------------------------------------------

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Cargando Llama 3.1 8B en 4 bits...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    quantization_config=quant_config,
    device_map="auto"
)

model = prepare_model_for_kbit_training(
    model
)

model.config.use_cache = False

# ------------------------------------------------------------
# LORA
# ------------------------------------------------------------

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

model = get_peft_model(
    model,
    lora_config
)

# ------------------------------------------------------------
# TOKENIZAR DATASET
# ------------------------------------------------------------

MAX_LENGTH = 512

def preparar_ejemplo(ejemplo):

    texto = tokenizer_llama.apply_chat_template(
        ejemplo["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    tokens = tokenizer_llama(
        texto,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens


dataset_tokenizado = dataset.map(
    preparar_ejemplo,
    remove_columns=dataset.column_names
)

print()
print("✓ Modelo QLoRA preparado")
print("✓ Dataset tokenizado")
print("Ejemplos:", len(dataset_tokenizado))

model.print_trainable_parameters()

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Cargando Llama 3.1 8B en 4 bits...


...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]


✓ Modelo QLoRA preparado
✓ Dataset tokenizado
Ejemplos: 6
trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848


# ETAPA 8 — Entrenamiento QLoRA

En esta etapa se ejecuta el entrenamiento de los adaptadores LoRA.

Se utilizan parámetros pequeños para reducir el consumo de memoria:

- batch size de 1;
- acumulación de gradientes;
- pocas épocas;
- precisión FP16;
- optimizador de 8 bits;
- sin checkpoints intermedios.


In [8]:
# ============================================================
# ETAPA 8 - ENTRENAMIENTO QLORA
# ============================================================

from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

OUTPUT_DIR = QLORA_DIR / "adaptador_isc"

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    num_train_epochs=3,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    fp16=True,

    logging_steps=1,
    save_strategy="no",

    report_to="none",

    optim="paged_adamw_8bit"
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer_llama,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_tokenizado,
    data_collator=data_collator
)

print("Iniciando entrenamiento QLoRA...")

trainer.train()

print()
print("✓ Entrenamiento QLoRA terminado")

Iniciando entrenamiento QLoRA...


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,4.086508
2,4.323183
3,4.108479
4,3.775892
5,3.160793
6,3.182228



✓ Entrenamiento QLoRA terminado


# ETAPA 9 — Guardar y probar el adaptador QLoRA

En esta etapa se:

- guarda el adaptador LoRA entrenado;
- guarda el tokenizer;
- realiza una prueba de generación;
- verifica que el modelo ajustado responda correctamente.

In [10]:
# ============================================================
# ETAPA 9 - GUARDAR Y PROBAR EL ADAPTADOR
# ============================================================

ADAPTER_DIR = QLORA_DIR / "adaptador_isc"

model.save_pretrained(
    ADAPTER_DIR
)

tokenizer_llama.save_pretrained(
    ADAPTER_DIR
)

print("✓ Adaptador guardado")
print("Ruta:", ADAPTER_DIR)

✓ Adaptador guardado
Ruta: /content/drive/MyDrive/Mx.Oscar.Hdez/qlora/adaptador_isc


In [11]:
# ------------------------------------------------------------
# PRUEBA DEL MODELO AJUSTADO
# ------------------------------------------------------------

model.eval()
model.config.use_cache = True

mensajes = [
    {
        "role": "user",
        "content": (
            "¿Qué debes hacer si no tienes "
            "información suficiente para responder?"
        )
    }
]

prompt = tokenizer_llama.apply_chat_template(
    mensajes,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer_llama(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():

    salida = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

respuesta = tokenizer_llama.decode(
    salida[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("=" * 60)
print("RESPUESTA DEL MODELO AJUSTADO")
print("=" * 60)

print(respuesta)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


RESPUESTA DEL MODELO AJUSTADO
Si no tengo información suficiente para responder a tu pregunta, puedo hacer varias cosas:

1. **Preguntar por más información**: Puedo pedirte que me proporciones más detalles o contexto sobre la pregunta para que pueda entender mejor el tema y proporcionarte una respuesta más precisa.
2. **Buscar en línea**: Puedo buscar en línea para encontrar información relevante y actualizada sobre el tema, aunque esto puede llevar un poco de tiempo.
3. **Propor
